In [2]:
import pandas as pd
import numpy as np

from sklearn import preprocessing
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics


from xailib.data_loaders.dataframe_loader import prepare_dataframe

from xailib.explainers.lime_explainer import LimeXAITabularExplainer
from xailib.explainers.lore_explainer import LoreTabularExplainer
from xailib.explainers.shap_explainer_tab import ShapXAITabularExplainer

from xailib.models.sklearn_classifier_wrapper import sklearn_classifier_wrapper

In [3]:
import pickle
import altair as alt

# Learning and explaining German Credit Dataset

## Loading and preparation of data

We start by reading from a CSV file the dataset to analyze. The table is loaded by means of the ```DataFrame``` class from the ```pandas``` library.

Among all the attributes of the table, we select the ```class_field``` column that contains the observed class for the corresponding row.

In [4]:
source_file = '../datasets/german_credit.csv'
class_field = 'default'
# Load and transform dataset 
df = pd.read_csv(source_file, skipinitialspace=True, na_values='?', keep_default_na=True)

After the data is loaded in memory, we need to extract metadata information to automatically handle the content withint the table.

The method ```prepare_dataframe``` scans the table and extract the following information:
 * ```df```: is a trasformed version of the original dataframe, where discrete attributes are transformed into numerical attributes by using one hot encoding strategy;
 * ```feature_names```: is a list containint the names of the features after the transformation;
 * ```class_values```: the list of all the possible values for the ```class_field``` column;
 * ```numeric_columns```: a list of the original features that contain numeric (i.e. continuous) values;
 * ```rdf```: the original dataframe, before the transformation;
 * ```real_feature_names```: the list of the features of the dataframe before the transformation;
 * ```features_map```: it is a dictionary pointing each feature to the original one before the transformation.

In [24]:
df, feature_names, class_values, numeric_columns, rdf, real_feature_names, features_map = prepare_dataframe(df, class_field)

### Learning a Random Forest classfier

We train a RF classifier by using the ```sklearn``` library. We start by splitting the dataset into a train and test subsets. 

In [6]:
test_size = 0.3
random_state = 42
X_train, X_test, Y_train, Y_test = train_test_split(df[feature_names], df[class_field],
                                                        test_size=test_size,
                                                        random_state=random_state,
                                                        stratify=df[class_field])



Then we train the model on the training set. 
Once the model has been learned, we use a wrapper class to get access to the model for ```XAI lib```

In [7]:
bb = RandomForestClassifier(n_estimators=20, random_state=random_state)
bb.fit(X_train.values, Y_train.values)
bbox = sklearn_classifier_wrapper(bb)   

In [8]:
y_prob = bb.predict_proba(X_test)[:,1] # This will give you positive class prediction probabilities  
y_pred = np.where(y_prob > 0.5, 1, 0) # This will threshold the probabilities to give class predictions.
bb.score(X_test, y_pred)

1.0

In [9]:
confusion_matrix=metrics.confusion_matrix(Y_test,y_pred)
confusion_matrix

array([[188,  22],
       [ 61,  29]])

Select a new instance to be classfied by the model and print the predicted class.

In [11]:
inst = X_train.iloc[8].values
print('Instance ',inst)
print('True class ',Y_train.iloc[8])
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [6 428 2 1 49 1 1 False True False False False False False True False
 False False False False False False False True False False False True
 False False False True False False False False True False False False
 False False True True False False False True False False False True False
 False True False False False True False True]
True class  0
Predicted class  [0]


In [25]:
real_inst = inst

## Explaining the prediction
We use the explanators of ```XAI lib``` to provide an explantion for the classified instance ```inst```.
Every explainer of ```XAI lib``` takes in input the blackbox to be explained with the corresponding feature names, and a configuration object to initialize the explainer.

### SHAP explainer

In [26]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'tree', 'X_train' : X_train.iloc[0:100].values}
explainer.fit(config)

In [27]:
exp = explainer.explain(inst)
print(exp.exp)

[array([ 9.08969237e-02,  3.54158791e-02,  9.04111037e-03,  7.12314474e-03,
        2.14812474e-02,  8.84168512e-03,  4.80699894e-04,  7.36877858e-03,
       -3.18269004e-02,  2.67333589e-03, -2.31245064e-02,  5.27087226e-03,
       -8.95955606e-03,  1.18502096e-03,  1.68228154e-02,  1.54729584e-04,
        4.61729892e-03,  2.91323560e-04,  6.27724705e-04, -2.02355914e-03,
       -6.84674572e-03, -5.93965012e-05, -2.25484695e-04, -1.20576181e-02,
        1.51080097e-03, -2.77883919e-05,  2.41358109e-04,  4.68100615e-03,
        3.32465702e-04,  1.26833632e-04,  2.09400522e-03,  7.34124956e-03,
        3.78453208e-03,  1.01209826e-02,  1.04533734e-04,  3.99039921e-03,
        1.51532677e-02,  1.12964852e-03,  2.75136030e-03, -7.33131963e-03,
        7.47411902e-04, -1.74160532e-03, -1.37533201e-03, -2.05295406e-02,
       -1.32863985e-03, -2.05483280e-02,  6.49798426e-03, -9.43885061e-03,
       -4.30130675e-02, -6.69614520e-03, -9.82695338e-04,  6.11445281e-03,
        1.01616604e-02, 

In [15]:
exp.plot_features_importance()

TypeError: type object got multiple values for keyword argument 'value'

### LORE explainer

In [28]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)

exp = explainer.explain(inst)
print(exp)

In [18]:
inst

array([6, 428, 2, 1, 49, 1, 1, False, True, False, False, False, False,
       False, True, False, False, False, False, False, False, False,
       False, True, False, False, False, True, False, False, False, True,
       False, False, False, False, True, False, False, False, False,
       False, True, True, False, False, False, True, False, False, False,
       True, False, False, True, False, False, False, True, False, True],
      dtype=object)

In [19]:
exp.expDict

{'bb_pred': 0,
 'dt_pred': 0,
 'rule': {'premise': [{'att': 'account_check_status=< 0 DM',
    'op': '>',
    'thr': 0.7868554294109344,
    'is_continuous': True},
   {'att': 'duration_in_month',
    'op': '<=',
    'thr': 17.5,
    'is_continuous': True},
   {'att': 'installment_as_income_perc',
    'op': '<=',
    'thr': 3.5,
    'is_continuous': True},
   {'att': 'present_emp_since=unemployed',
    'op': '<=',
    'thr': 0.5,
    'is_continuous': True},
   {'att': 'credit_amount', 'op': '>', 'thr': -476.5, 'is_continuous': True}],
  'cons': 0,
  'class_name': 'default'},
 'crules': [{'premise': [{'att': 'account_check_status=< 0 DM',
     'op': '>',
     'thr': 0.01842014119029045,
     'is_continuous': True},
    {'att': 'duration_in_month',
     'op': '>',
     'thr': 25.0,
     'is_continuous': True},
    {'att': 'account_check_status=no checking account',
     'op': '>',
     'thr': 0.8679453730583191,
     'is_continuous': True},
    {'att': 'installment_as_income_perc',
     

In [20]:
rules_dict = open("rules_dict_two.pkl", "wb")
pickle.dump(exp.expDict, rules_dict)
rules_dict.close()

In [25]:
pd.read_pickle(r'rules_dict_two.pkl')

{'bb_pred': 0,
 'dt_pred': 0,
 'rule': {'premise': [{'att': 'account_check_status=< 0 DM',
    'op': '>',
    'thr': 0.01842014119029045,
    'is_continuous': True},
   {'att': 'duration_in_month',
    'op': '<=',
    'thr': 18.16944408416748,
    'is_continuous': True},
   {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
    'op': '>',
    'thr': 0.5962575227022171,
    'is_continuous': True}],
  'cons': 0,
  'class_name': 'default'},
 'crules': [{'premise': [{'att': 'account_check_status=< 0 DM',
     'op': '<=',
     'thr': 0.01842014119029045,
     'is_continuous': True},
    {'att': 'duration_in_month',
     'op': '>',
     'thr': 23.5,
     'is_continuous': True},
    {'att': 'housing=own',
     'op': '>',
     'thr': 0.8447267115116119,
     'is_continuous': True},
    {'att': 'credit_history=no credits taken/ all credits paid back duly',
     'op': '<=',
     'thr': 0.0690189003944397,
     'is_continuous': True},
    {'att': 'installment_as

In [26]:
exp.plotRules()

In [27]:
exp.exp

In [459]:
exp.plotCounterfactualRules()

### LIME explainer

In [460]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())

[('account_check_status=no checking account', -0.031955012192442574), ('duration_in_month', 0.030839782539954504), ('account_check_status=< 0 DM', 0.027467865053919998), ('credit_history=critical account/ other credits existing (not at this bank)', -0.0264345939322612), ('other_installment_plans=bank', 0.02294961869095157), ('age', -0.02217886749081485), ('property=real estate', -0.02017192665864926), ('savings=... < 100 DM', 0.01793407873368487), ('installment_as_income_perc', 0.015581579840944697), ('property=unknown / no property', 0.015253540546335102)]


In [461]:
# limeExplainer.plot_lime_values(lime_exp.as_list(), 5, 10)
lime_exp.plot_features_importance()

alt.VConcatChart(...)

## Learning a different model

### Learning a Logistic Regressor

We train a Logistic Regression by using the ```sklearn``` library. We transform the dataset by using a ```Scaler``` to normalize all the attributes.


In [462]:
scaler = preprocessing.StandardScaler().fit(X_train)
X_scaled = scaler.transform(X_train)

bb = LogisticRegression(C=1, penalty='l2')
bb.fit(X_scaled, Y_train.values)
# pass the model to the wrapper to use it in the XAI lib
bbox = sklearn_classifier_wrapper(bb)

In [463]:
# select a record to explain
inst = X_scaled[182]
print('Instance ',inst)
print('Predicted class ',bb.predict(inst.reshape(1, -1)))

Instance  [ 2.27797454  3.35504085  0.94540357  1.07634233  0.04854891 -0.72456474
 -0.43411405  1.65027399 -0.61477862 -0.25898489 -0.80681063  4.17385345
 -0.6435382  -0.32533856 -1.03489416 -0.20412415 -0.22941573 -0.33068147
  1.75885396 -0.34899122 -0.60155441 -0.15294382 -0.09298136 -0.46852129
 -0.12038585 -0.08481889 -0.23623492 -1.21387736 -0.36174054 -0.24943031
  2.15526362 -0.59715086 -0.45485883 -0.73610476 -0.43875307  4.23307441
 -0.65242771 -0.23958675 -0.32533856  0.90192655  4.72581563 -0.2259448
 -3.15238005 -0.54212562 -0.70181003 -0.63024248  2.30354212 -0.40586384
  0.49329429 -0.23958675  2.88675135 -1.59227935 -0.46170508  2.46388049
 -1.33747696 -0.13206764 -0.5        -1.21387736  1.21387736 -0.20412415
  0.20412415]
Predicted class  [1]


In [495]:
X_scaled

array([[-0.7335121 , -0.71300074,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766, -0.61086948,  0.0547138 , ..., -0.82380645,
        -0.20412415,  0.20412415],
       [-0.23159766,  0.3606869 , -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415],
       ...,
       [-0.23159766, -0.25658997,  0.0547138 , ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.23159766,  1.97159246, -1.72666575, ...,  1.21387736,
        -0.20412415,  0.20412415],
       [-0.48255488, -0.06429887, -0.83597597, ..., -0.82380645,
        -0.20412415,  0.20412415]])

## Explaining the prediction
We use the same explainators as for the previous model. In this case, a few adjustments are necessary for the initialization of the explanators. For example, SHAP needs a specific configuration for the linear model we are using.
### SHAP Explainer

In [464]:
explainer = ShapXAITabularExplainer(bbox, feature_names)
config = {'explainer' : 'linear', 'X_train' : X_scaled[0:100], 'feature_pert' : 'interventional'}
explainer.fit(config)

In [465]:
exp = explainer.explain(inst)
print(exp)

In [466]:
exp.plot_features_importance()

alt.VConcatChart(...)

### LORE explainer

In [467]:
explainer = LoreTabularExplainer(bbox)
config = {'neigh_type':'geneticp', 'size':1000, 'ocr':0.1, 'ngen':10}
explainer.fit(df, class_field, config)
exp = explainer.explain(inst)
print(exp)

In [468]:
exp.plotRules()

In [469]:
exp.plotCounterfactualRules()

### LIME explainer

In [470]:
limeExplainer = LimeXAITabularExplainer(bbox)
config = {'feature_selection': 'lasso_path'}
limeExplainer.fit(df, class_field, config)
lime_exp = limeExplainer.explain(inst)
print(lime_exp.exp.as_list())

[('other_debtors=co-applicant', -1.2099484483255133e-09), ('credit_history=all credits at this bank paid back duly', -9.807292855860103e-10), ('present_emp_since=unemployed', -8.905493791597458e-10), ('other_debtors=none', 6.501490037885442e-10), ('housing=for free', -4.510074423165472e-10), ('credit_amount', 3.4347399311566505e-10), ('job=management/ self-employed/ highly qualified employee/ officer', -3.0720840863891935e-10), ('property=unknown / no property', -3.047260454092378e-10), ('savings=unknown/ no savings account', -3.0122747825365293e-10), ('savings=... < 100 DM', 2.4055110623915944e-10), ('credits_this_bank', 2.3718772642551254e-10), ('housing=own', 2.1981825790192012e-10), ('property=real estate', 1.9425271631666957e-10), ('account_check_status=0 <= ... < 200 DM', -1.8341777236604516e-10), ('foreign_worker=yes', 1.7351459527660452e-10), ('purpose=car (new)', -1.6299442716105333e-10), ('account_check_status=no checking account', 1.622711779010775e-10), ('people_under_maint

In [471]:
lime_exp.plot_features_importance()

alt.VConcatChart(...)

In [472]:
rules =exp.expDict['rule']['premise']

In [473]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [474]:
for r in rules:
    print(r['att'])

age
credit_amount
purpose=retraining
duration_in_month
purpose=furniture/equipment
foreign_worker=no
purpose=domestic appliances
savings=.. >= 1000 DM 
purpose=(vacation - does not exist?)
credit_history=critical account/ other credits existing (not at this bank)
people_under_maintenance


In [475]:
df_range=pd.concat({'min':X_train.min(), 'max':X_train.max()},axis=1)

In [476]:
df_range=df_range.reset_index()

In [477]:
df_range

,index,min,max
0,duration_in_month,4,60
1,credit_amount,338,15945
2,installment_as_income_perc,1,4
3,present_res_since,1,4
4,age,19,75
...,...,...,...
56,job=unskilled - resident,0,1
57,telephone=none,0,1
58,"telephone=yes, registered under the customers ...",0,1
59,foreign_worker=no,0,1


In [478]:
rules

[{'att': 'age', 'op': '<=', 'thr': 20.726173400878906, 'is_continuous': True},
 {'att': 'credit_amount',
  'op': '>',
  'thr': -439.6443485021591,
  'is_continuous': True},
 {'att': 'purpose=retraining',
  'op': '<=',
  'thr': 0.11524588242173195,
  'is_continuous': True},
 {'att': 'duration_in_month',
  'op': '>',
  'thr': -1.9407005310058594,
  'is_continuous': True},
 {'att': 'purpose=furniture/equipment',
  'op': '<=',
  'thr': 0.18370826542377472,
  'is_continuous': True},
 {'att': 'foreign_worker=no',
  'op': '<=',
  'thr': 0.7168410122394562,
  'is_continuous': True},
 {'att': 'purpose=domestic appliances',
  'op': '<=',
  'thr': 1.015466570854187,
  'is_continuous': True},
 {'att': 'savings=.. >= 1000 DM ',
  'op': '<=',
  'thr': 0.7176859378814697,
  'is_continuous': True},
 {'att': 'purpose=(vacation - does not exist?)',
  'op': '<=',
  'thr': 0.4622504562139511,
  'is_continuous': True},
 {'att': 'credit_history=critical account/ other credits existing (not at this bank)',
 

In [479]:
df_rules = pd.DataFrame.from_records(rules)
df_rules

,att,op,thr,is_continuous
0,age,<=,20.726173,True
1,credit_amount,>,-439.644349,True
2,purpose=retraining,<=,0.115246,True
3,duration_in_month,>,-1.940701,True
4,purpose=furniture/equipment,<=,0.183708,True
5,foreign_worker=no,<=,0.716841,True
6,purpose=domestic appliances,<=,1.015467,True
7,savings=.. >= 1000 DM,<=,0.717686,True
8,purpose=(vacation - does not exist?),<=,0.462250,True
9,credit_history=critical account/ other credits...,<=,0.908596,True


In [480]:
df_viz = df_range.merge(df_rules,how='left',left_on='index',right_on='att')
df_viz = df_viz.drop('att', axis=1)
df_viz

,index,min,max,op,thr,is_continuous
0,duration_in_month,4,60,>,-1.940701,True
1,credit_amount,338,15945,>,-439.644349,True
2,installment_as_income_perc,1,4,NaN,NaN,NaN
3,present_res_since,1,4,NaN,NaN,NaN
4,age,19,75,<=,20.726173,True
...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN
57,telephone=none,0,1,NaN,NaN,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN
59,foreign_worker=no,0,1,<=,0.716841,True


array([ 2.27797454,  3.35504085,  0.94540357,  1.07634233,  0.04854891,
       -0.72456474, -0.43411405,  1.65027399, -0.61477862, -0.25898489,
       -0.80681063,  4.17385345, -0.6435382 , -0.32533856, -1.03489416,
       -0.20412415, -0.22941573, -0.33068147,  1.75885396, -0.34899122,
       -0.60155441, -0.15294382, -0.09298136, -0.46852129, -0.12038585,
       -0.08481889, -0.23623492, -1.21387736, -0.36174054, -0.24943031,
        2.15526362, -0.59715086, -0.45485883, -0.73610476, -0.43875307,
        4.23307441, -0.65242771, -0.23958675, -0.32533856,  0.90192655,
        4.72581563, -0.2259448 , -3.15238005, -0.54212562, -0.70181003,
       -0.63024248,  2.30354212, -0.40586384,  0.49329429, -0.23958675,
        2.88675135, -1.59227935, -0.46170508,  2.46388049, -1.33747696,
       -0.13206764, -0.5       , -1.21387736,  1.21387736, -0.20412415,
        0.20412415])

In [502]:
df_viz['inst'] = real_inst.tolist()
df_viz

,index,min,max,op,thr,is_continuous,inst,thr2
0,duration_in_month,4,60,>,-1.940701,True,15,60.0
1,credit_amount,338,15945,>,-439.644349,True,975,15945.0
2,installment_as_income_perc,1,4,NaN,NaN,NaN,2,NaN
3,present_res_since,1,4,NaN,NaN,NaN,3,NaN
4,age,19,75,<=,20.726173,True,25,19.0
...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN,0,NaN
57,telephone=none,0,1,NaN,NaN,NaN,1,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN,0,NaN
59,foreign_worker=no,0,1,<=,0.716841,True,0,0.0


In [503]:
thr2_list=[]
for i, row in df_viz.iterrows():
    if (row['op']=='>' or row['op']== '>='):
        thr2_list.append(row['max'])
        continue
    if (row['op']=='<' or row['op']== '<='):
        thr2_list.append(row['min'])
        continue
    else:
        thr2_list.append(np.nan)
df_viz['thr2'] = thr2_list
df_viz

,index,min,max,op,thr,is_continuous,inst,thr2
0,duration_in_month,4,60,>,-1.940701,True,15,60.0
1,credit_amount,338,15945,>,-439.644349,True,975,15945.0
2,installment_as_income_perc,1,4,NaN,NaN,NaN,2,NaN
3,present_res_since,1,4,NaN,NaN,NaN,3,NaN
4,age,19,75,<=,20.726173,True,25,19.0
...,...,...,...,...,...,...,...,...
56,job=unskilled - resident,0,1,NaN,NaN,NaN,0,NaN
57,telephone=none,0,1,NaN,NaN,NaN,1,NaN
58,"telephone=yes, registered under the customers ...",0,1,NaN,NaN,NaN,0,NaN
59,foreign_worker=no,0,1,<=,0.716841,True,0,0.0


In [504]:
features=df_viz['index'].to_list()

In [511]:
ch_list=[]
for i, row in df_viz.iterrows():
    if row['inst']!=0 or row['is_continuous']==True:
        p=alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_rule(
            color='red' if row['is_continuous'] == True else'black',
            size=2
        ).encode(
            x=alt.X(
                field='inst',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            tooltip=[alt.Tooltip(field='inst',title=row['index'])]
        )
        
        t_min = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='black',
            dx=-10,
            align='right',
#             fontWeight='bold'
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None
            ),
            text='min:N'
        )

        t_max = alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_text(
            color='grey',
            dx=5,
            align='left',
            fontWeight='bold'
        ).encode(
            x=alt.X(
                field='max',
                type='quantitative',
                title=None
            ),
            text='max:N'
        )
            
        b =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='#f4dd4d',size=5
        ).encode(
            x=alt.X(
                field='thr',
                type='quantitative',
                title=None,
            ),
            x2='thr2',
            y=alt.Y(field='index',type='nominal',title=None),

        )
        
        l =alt.Chart(
            df_viz[df_viz['index']==row['index']]
        ).mark_bar(
            color='grey',size=1
        ).encode(
            x=alt.X(
                field='min',
                type='quantitative',
                title=None,
                scale= alt.Scale(domain=(row['min'],row['max']),clamp=True,nice=False)
            ),
            x2='max',
            y=alt.Y(field='index',type='nominal',title=None),
        )
        
        
        comp = alt.layer(l,b,t_min,t_max,p).properties(
            height=10,
            width=100
        )
        ch_list.append(comp)
concat=alt.vconcat(*ch_list,  title=f"Predicted class: {exp.expDict['bb_pred']}")

concat.configure_concat(
    spacing=0
).configure_axis(
    grid=False
).configure_view(
    strokeWidth=1
).configure_axisX(
    disable=True
).configure_axisY(
    domain=False,
    ticks=False,
    labelPadding=50,
    minExtent=300
).configure_title(
    fontWeight='bold', anchor="start"
)

alt.VConcatChart(...)

# todo:

- Riscrivere il codice in una funzione ammodino
- Check su cosa prendere per le regole e le soglie
- Usare la FI per ordinare le feature 
- hconcat con la FI 
- aggiungere il cutoff su entrambi
- inserire progressive disclosure:
    - filtrare per feature a cui è associata Rules
    - filtrare per FI (cutoff)
- Inserire nella funzione per plottare il preprocessing dei dati 

- Investigare le CR
- Fare la stessa cosa con titanic

- nel paper partire dalla vecchia viz html (linguaggio naturale)